<a href="https://colab.research.google.com/github/Abraham1439/Rag_Reglamento_Biblioteca/blob/main/Rag_Reglamento_Biblioteca_Duoc.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Asistente Virtual RAG de Biblioteca Duoc UC

Este notebook reúne todo el código del pipeline RAG del proyecto **Biblioteca Duoc UC** en un solo archivo ejecutable directamente en **Google Colab**.

### Arquitectura del Notebook:
1. **Instalación de Dependencias**: `openai`, `langchain`, `faiss-cpu`, etc.
2. **Configuración y Claves API**: Groq (LLM) y Mistral (Embeddings).
3. **Carga y Fragmentación (Ingesta)**: Creación de documentos del reglamento y chunking.
4. **Índice Vectorial (FAISS)**: Generación de embeddings e indexación.
5. **Prompt Engineering**: Estrategias Zero-Shot, Few-Shot y Chain-of-Thought.
6. **Agente RAG + Filtro de Confianza**: Lógica de respuesta y derivación.
7. **Evaluación de Casos**: Pruebas con consultas dentro y fuera de alcance.
8. **Comparativa de Técnicas de Prompting**: Benchmark empírico de Zero-Shot, Few-Shot y Chain-of-Thought.

## 1. Instalación de Dependencias

In [ ]:
!pip install -q openai langchain langchain-core langchain-openai langchain-community langchain-text-splitters faiss-cpu python-dotenv numpy

## 2. Configuración del Entorno y Claves API

In [ ]:
import os
import getpass

# --- Cargar Credenciales (Google Colab Secrets o .env Local) ---
try:
    from google.colab import userdata          # Entra aquí si estás en Google Colab

    # Lista de variables a buscar en los Secrets de Colab
    colab_keys = (
        "LLM_API_KEY", "LLM_BASE_URL", "LLM_MODEL",
        "EMBEDDING_API_KEY", "EMBEDDING_BASE_URL", "EMBEDDING_MODEL"
    )
    for key in colab_keys:
        try:
            val = userdata.get(key)
            if val:
                os.environ[key] = val
        except Exception:
            pass                                # Si el Secret no existe en Colab, no hace nada
except ImportError:
    from dotenv import load_dotenv             # Entra aquí si estás corriendo en tu máquina local
    load_dotenv()

# --- Configuración por defecto si no están definidas en entorno/secrets ---
os.environ.setdefault("LLM_BASE_URL", "https://api.groq.com/openai/v1")
os.environ.setdefault("LLM_MODEL", "groq/compound-mini")

os.environ.setdefault("EMBEDDING_BASE_URL", "https://api.mistral.ai/v1")
os.environ.setdefault("EMBEDDING_MODEL", "mistral-embed")

# --- Asignación de Variables ---
LLM_BASE_URL = os.getenv("LLM_BASE_URL")
LLM_MODEL = os.getenv("LLM_MODEL")
# Busca en Secrets/env; si no existe, te pedirá ingresarla interactivamente como respaldo
LLM_API_KEY = os.getenv("LLM_API_KEY") or getpass.getpass("Ingresa tu LLM_API_KEY (Groq): ")

EMBEDDING_BASE_URL = os.getenv("EMBEDDING_BASE_URL")
EMBEDDING_MODEL = os.getenv("EMBEDDING_MODEL")
EMBEDDING_API_KEY = os.getenv("EMBEDDING_API_KEY") or getpass.getpass("Ingresa tu EMBEDDING_API_KEY (Mistral): ")

# --- Configuración del Proyecto ---
from pathlib import Path

def localizar_data(inicio=None):
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio.resolve(), *inicio.resolve().parents):
        if (candidato / "notebook" / "asistente_biblioteca_duoc.ipynb").is_file():
            carpeta = candidato / "data"
            if not carpeta.is_dir():
                raise FileNotFoundError(f"Falta la carpeta de datos del proyecto: {carpeta}")
            return carpeta
    raise FileNotFoundError(
        "Abre el notebook desde la carpeta del proyecto o desde su subcarpeta notebook. "
        "En Colab, carga el proyecto completo conservando data/ y notebook/."
    )

DATA_DIR = str(localizar_data())
CHUNK_SIZE = 500
CHUNK_OVERLAP = 80
TOP_K = 3
CONFIDENCE_THRESHOLD = 0.45

print("Configuración cargada correctamente:")
print(f"- LLM Model: {LLM_MODEL} (Groq)")
print(f"- Embedding Model: {EMBEDDING_MODEL} (Mistral)")

## 3. Validación de los documentos existentes

Se utiliza únicamente `data/` en la raíz del proyecto, al mismo nivel que `notebook/`. La ruta se resuelve tanto al ejecutar desde la raíz como desde `notebook/`. No se crean ni se sobrescriben documentos. Todos las reglas de la biblioteca se encuentran dentro de la carpeta llamada data.


In [ ]:
# Comprobar los documentos de la carpeta data/ del proyecto sin modificarlos.
ARCHIVOS_REQUERIDOS = (
    "reglamento_general.txt", "prestamos_renovaciones.txt", "morosos_sanciones.txt",
    "normas_disciplinarias.txt", "salas_estudio.txt", "uso_lentes_vr.txt",
)
faltantes = [nombre for nombre in ARCHIVOS_REQUERIDOS if not (Path(DATA_DIR) / nombre).is_file()]
if faltantes:
    raise FileNotFoundError(f"Faltan documentos en {DATA_DIR}: {', '.join(faltantes)}")
print(f"Se leerán {len(ARCHIVOS_REQUERIDOS)} documentos existentes desde: {DATA_DIR}")


## 4. Ingesta: Carga, Fragmentación y Construcción del Índice Vectorial (FAISS)

In [ ]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings

DOCUMENT_TYPES = {
    "reglamento_general.txt": "reglamento_general",
    "prestamos_renovaciones.txt": "prestamos",
    "morosos_sanciones.txt": "sanciones",
    "normas_disciplinarias.txt": "normas_disciplinarias",
    "salas_estudio.txt": "salas_estudio",
    "uso_lentes_vr.txt": "lentes_vr",
}

def load_documents(data_dir: str = DATA_DIR) -> list[Document]:
    documents = []
    for filename, doc_type in DOCUMENT_TYPES.items():
        path = os.path.join(data_dir, filename)
        if not os.path.exists(path):
            raise FileNotFoundError(f"No se encontró el documento requerido: {path}")
        with open(path, "r", encoding="utf-8") as f:
            text = f.read()
        documents.append(
            Document(page_content=text, metadata={"tipo": doc_type, "fuente": filename})
        )
    return documents

def split_documents(documents: list[Document], chunk_size: int = CHUNK_SIZE, chunk_overlap: int = CHUNK_OVERLAP) -> list[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\nArtículo", "\n\n", "\n", ". ", " ", ""],
    )
    return splitter.split_documents(documents)

def build_embeddings():
    return OpenAIEmbeddings(
        base_url=EMBEDDING_BASE_URL,
        api_key=EMBEDDING_API_KEY,
        model=EMBEDDING_MODEL,
        check_embedding_ctx_length=False,
    )

# --- Ejecutar la ingesta ---
docs = load_documents()
chunks = split_documents(docs)
embeddings = build_embeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)

print(f"Índice FAISS construido exitosamente con {vectorstore.index.ntotal} chunks.")

## 5. Formulación de Prompts (Prompts Engineering)

In [ ]:
SYSTEM_INSTRUCTIONS = """Eres el Asistente Virtual de la Biblioteca Duoc UC (sede Plaza Norte).
Respondes exclusivamente con base en el CONTEXTO entregado, que proviene del
Reglamento de Bibliotecas Duoc UC y de información operativa de la sede.

Reglas estrictas:
1. No inventes plazos, montos de multas, artículos ni condiciones que no estén explícitos en el CONTEXTO.
2. Si el CONTEXTO no contiene información suficiente para responder con seguridad, indica que no cuentas con esa información y que se dirija con su consulta a un encargado de biblioteca. No intentes adivinar.
3. Cuando cites una norma, menciona el número de artículo si aparece en el CONTEXTO (ej. "según el Artículo 15°...").
4. Responde en español, de forma breve, clara y cordial.
"""

DERIVATION_MESSAGE = (
    "No tengo información suficiente y verificada en el reglamento para responder "
    "esa consulta con seguridad. Por favor, dirígete a un encargado de "
    "Biblioteca Duoc UC para que te ayude directamente."
)

FEW_SHOT_EXAMPLES = [
    {
        "pregunta": "¿Cuántos días puedo mantener un libro de la colección general?",
        "contexto": "Colección General: tiempo de préstamo de 7 días, con un límite de 10 renovaciones por semestre.",
        "respuesta": "Según el Artículo 15°, un libro de la Colección General se presta por 7 días, con hasta 10 renovaciones por semestre.",
    },
    {
        "pregunta": "¿Puedo pedir un descuento en la multa si soy buen alumno?",
        "contexto": "El usuario que se encuentre en mora se hará acreedor de una multa de $1.000 por cada 7 días de atraso, por ítem.",
        "respuesta": DERIVATION_MESSAGE,
    },
]

def build_prompt(context: str, question: str, technique: str = "zero-shot") -> str:
    if technique == "zero-shot":
        return f"{SYSTEM_INSTRUCTIONS}\n\nCONTEXTO:\n{context}\n\nPREGUNTA DEL ESTUDIANTE:\n{question}\n\nRESPUESTA:"
    elif technique == "few-shot":
        ejemplos = "\n\n".join(
            f"Pregunta: {ej['pregunta']}\nContexto: {ej['contexto']}\nRespuesta: {ej['respuesta']}"
            for ej in FEW_SHOT_EXAMPLES
        )
        return f"{SYSTEM_INSTRUCTIONS}\n\nEjemplos:\n{ejemplos}\n\nNUEVO CASO:\nCONTEXTO:\n{context}\n\nPREGUNTA:\n{question}\n\nRESPUESTA:"
    elif technique == "chain-of-thought":
        return f"{SYSTEM_INSTRUCTIONS}\n\nCONTEXTO:\n{context}\n\nPREGUNTA:\n{question}\n\nRazona paso a paso qué artículos aplican y entrega SOLO la respuesta final al estudiante.\n\nRESPUESTA:"
    else:
        raise ValueError(f"Técnica desconocida: {technique}")